In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch

In [ ]:
def resample(x: pq.Audio, ratio: float) -> pq.Audio:
    """Read x at `ratio` times its original speed, with linear interpolation."""
    samples = np.asarray(x.samples)
    N = len(samples)
    M = int(N / ratio)                        # the duration scales by 1 / ratio
    p = np.arange(M) * ratio                  # fractional read positions in x
    p_floor = np.floor(p).astype(int)
    alpha = (p - p_floor)[:, np.newaxis]
    p_next = np.minimum(p_floor + 1, N - 1)
    y = (1 - alpha) * samples[p_floor] + alpha * samples[p_next]
    return pq.Audio(y, x.sample_rate)


def repitch(x: pq.Audio, k: float) -> pq.Audio:
    """Shift the pitch of x by k steps."""
    return resample(x, 2 ** (k / 12))


def clarinet(pitch: str) -> pq.Audio:
    """Play any pitch by repitching a single recording of a clarinet playing A#5."""
    recording = pq.Audio.from_file("../assets/clarinet.wav")
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch("A#5")
    return repitch(recording, k)


arpeggio = pq.Score([(0.5 * i, {"pitch": pitch})
                     for i, pitch in enumerate(["A#4", "D5", "F5", "A#5", "D6", "F6"])])
pq.play(arpeggio.render(clarinet), normalize=True)